# Assignment 2: From Trees to Neural Networks

**Dataset:** UCI Bank Marketing (`bank-additional-full.csv`)

**Objective:** Compare gradient-boosted decision trees (XGBoost) and a multi-layer perceptron (MLP) on a tabular binary classification task: predict whether a client subscribes to a term deposit (`y = "yes"` is the positive class).

This notebook covers Step 1 — data preparation (audit, cleaning decisions, a stratified 70/15/15 split, preprocessing fitted on training data only), Step 2 — gradient-boosted decision trees with controlled tuning and early stopping, and Step 3 — a multi-layer perceptron with controlled tuning.

**How to run in Google Colab**
1. Upload this notebook to Colab.
2. Upload `bank-additional-full.csv` into the Colab working directory (left sidebar → Files → Upload).
3. Runtime → Run all.

## 0. Setup

In [ ]:
import os
import platform
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import average_precision_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_FILE = "bank-additional-full.csv"
TARGET = "y"
POSITIVE_LABEL = "yes"
LEAKAGE_COLUMNS = ["duration"]
PDAYS_NO_CONTACT = 999
PDAYS_NO_CONTACT_ENCODED = -1

print(f"Python:       {platform.python_version()}")
print(f"pandas:       {pd.__version__}")
print(f"numpy:        {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"xgboost:      {xgb.__version__}")
print(f"matplotlib:   {matplotlib.__version__}")
print(f"Platform:     {platform.platform()}")
print(f"CPU cores:    {os.cpu_count()}")
print(f"RANDOM_STATE: {RANDOM_STATE}")

### 0.1 Load the data

Place `bank-additional-full.csv` in the same working directory as this notebook. Source: UCI Machine Learning Repository, Bank Marketing dataset — `bank-additional-full.csv` from `bank-additional.zip`. The file is semicolon-separated.

In [ ]:
DATA_PATH = Path(DATA_FILE)
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_FILE} not found in the working directory. "
        "In Colab: Files → Upload → select bank-additional-full.csv, then re-run this cell."
    )

print(f"Found: {DATA_PATH.resolve()}")

## 1. Data audit

In [ ]:
df = pd.read_csv(DATA_PATH, sep=";")
print(f"Rows: {df.shape[0]:,}  Columns: {df.shape[1]}")
df.head()

### 1.1 Feature types

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = [c for c in df.columns if c not in numeric_cols and c != TARGET]

feature_types = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
}).drop(index=TARGET)
feature_types["type"] = np.where(feature_types.index.isin(numeric_cols), "numeric", "nominal categorical")

print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
feature_types

### 1.2 Target distribution

In [ ]:
target_counts = df[TARGET].value_counts()
target_share = df[TARGET].value_counts(normalize=True)
pd.DataFrame({"count": target_counts, "proportion": target_share.round(4)})

### 1.3 Missing and special values

In [ ]:
# pandas nulls vs. the dataset's literal "unknown" category
null_counts = df.isna().sum()
print(f"Total pandas-null cells: {int(null_counts.sum())}")

unknown_summary = pd.DataFrame({
    "unknown_count": [(df[c] == "unknown").sum() for c in categorical_cols],
}, index=categorical_cols)
unknown_summary["unknown_pct"] = (100 * unknown_summary["unknown_count"] / len(df)).round(2)
unknown_summary.sort_values("unknown_count", ascending=False)

In [ ]:
no_contact = df["pdays"] == PDAYS_NO_CONTACT
print(f"pdays == {PDAYS_NO_CONTACT}: {no_contact.sum():,} rows ({no_contact.mean():.2%})")
print("pdays range when a previous contact exists:",
      df.loc[~no_contact, "pdays"].min(), "to", df.loc[~no_contact, "pdays"].max())
print()
print(pd.crosstab(no_contact.rename("pdays == 999"), df["poutcome"]))

In [ ]:
print(f"Exact duplicate rows (all columns):           {df.duplicated().sum():,}")
print(f"Duplicate rows after excluding duration:      {df.drop(columns=LEAKAGE_COLUMNS).duplicated().sum():,}")
print(f"Rows where default == 'yes':                  {(df['default'] == 'yes').sum():,}")

**Audit findings.**

- The file has 41,188 rows and 21 columns: 10 numeric predictors, 10 nominal categorical predictors, and the target `y`.
- The positive class is rare: about 11.3% of clients subscribed. Accuracy alone would be misleading (predicting "no" for everyone scores about 88.7%), which is why Average Precision is the primary metric.
- There are no pandas-null values. Missingness appears instead as the literal category `"unknown"` in six columns, most heavily in `default` (about 20.9% of rows).
- Whether `"unknown"` carries signal about the target is checked on training rows only, after the split (Section 3.1).
- `pdays = 999` is the dataset's sentinel for *not previously contacted*, not 999 elapsed days, and covers about 96.3% of rows. Real values range from 0 to 27 days. Every `poutcome = "success"` row has a real `pdays` and every `"nonexistent"` row has `pdays = 999`, but 4,110 `"failure"` rows (with `previous > 0`) also have `pdays = 999`. In this file, 999 therefore means *no recorded days since a previous-campaign contact*, not always *no previous contact*.
- There are 12 exact duplicate rows. Without an identifier there is no way to tell whether they are repeated records or different clients with identical attributes.

## 2. Cleaning decisions and target definition

| Item | Decision | Reason |
|---|---|---|
| Target | `y = "yes"` → 1, `"no"` → 0 | Positive class defined by the handout |
| `duration` | Dropped from predictors | Call length is only known after the call ends; using it would leak information unavailable at prediction time |
| `pdays = 999` | New indicator `pdays_recorded` (1 when `pdays` holds a real day count); `pdays` set to −1 when it is 999 | 999 is a no-previous-contact sentinel, not an elapsed number of days. Leaving it would put these clients far from real values (0–27) and dominate the scale of the column. Because some rows with 999 still have `previous > 0`, the indicator is named for what it measures — whether a day count is recorded — while the number of previous contacts stays in `previous`. −1 sits just outside the valid range so both models can separate the two cases |
| Literal `"unknown"` | Kept as its own category and one-hot encoded | It is a recorded value, not a blank cell, and on training rows its subscription rate differs from known values (Section 3.1). Imputing the mode would erase that signal and invent information |
| Missing values (NaN) | No imputation needed | The audit found no pandas-null values |
| Duplicates | All rows kept | Only 12 exact duplicates (0.03%) and no client ID to confirm they are repeats; keeping all rows preserves the dataset described in the handout |
| `default = "yes"` | Kept | Only 3 rows; the encoder ignores the level if it never appears in training |

These steps are deterministic, row-by-row rules that learn nothing from the data, so applying them before the split does not leak information.

In [ ]:
def prepare_features(raw: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Deterministic, row-wise cleaning only (no statistics learned from the data)."""
    data = raw.copy()
    valid_labels = {"no", POSITIVE_LABEL}
    invalid = ~data[TARGET].isin(valid_labels)
    if invalid.any():
        raise ValueError(f"{TARGET} must be one of {sorted(valid_labels)}; found "
                         f"{data.loc[invalid, TARGET].unique().tolist()} in {int(invalid.sum())} rows")
    y = (data[TARGET] == POSITIVE_LABEL).astype(int).rename("target")

    data["pdays_recorded"] = (data["pdays"] != PDAYS_NO_CONTACT).astype(int)
    data["pdays"] = data["pdays"].where(data["pdays_recorded"] == 1, PDAYS_NO_CONTACT_ENCODED)

    X = data.drop(columns=[TARGET, *LEAKAGE_COLUMNS])
    return X, y


X, y = prepare_features(df)

assert "duration" not in X.columns
assert TARGET not in X.columns
assert (X.loc[X["pdays_recorded"] == 0, "pdays"] == PDAYS_NO_CONTACT_ENCODED).all()

print(f"X: {X.shape}  y: {y.shape}  positive rate: {y.mean():.4f}")
X[["pdays", "pdays_recorded", "previous", "poutcome"]].head()

In [ ]:
CATEGORICAL_FEATURES = [
    "job", "marital", "education", "default", "housing", "loan",
    "contact", "month", "day_of_week", "poutcome",
]
NUMERIC_FEATURES = [
    "age", "campaign", "pdays", "previous", "pdays_recorded",
    "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed",
]

assert set(CATEGORICAL_FEATURES) | set(NUMERIC_FEATURES) == set(X.columns)
assert not set(CATEGORICAL_FEATURES) & set(NUMERIC_FEATURES)
print(f"{len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical = {X.shape[1]} predictors")

## 3. Stratified 70/15/15 train / validation / test split

The split is made in two stages with `random_state = 42`, stratified on the target each time so all three partitions keep the same positive-class rate.

- **Train (70%)** — fit preprocessing and models.
- **Validation (15%)** — XGBoost early stopping and choosing configurations by validation Average Precision.
- **Test (15%)** — used once, after both final configurations are fixed.

The same split rows are reused for every GBDT and MLP experiment.

In [ ]:
TRAIN_SIZE, VALID_SIZE, TEST_SIZE = 0.70, 0.15, 0.15

# Stage 1: 70% train vs. 30% holdout. Stage 2: split the holdout 50/50 into validation and test.
X_train, X_hold, y_train, y_hold = train_test_split(
    X, y, train_size=TRAIN_SIZE, stratify=y, random_state=RANDOM_STATE
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_hold, y_hold, test_size=TEST_SIZE / (VALID_SIZE + TEST_SIZE), stratify=y_hold, random_state=RANDOM_STATE
)

split_summary = pd.DataFrame({
    "rows": [len(y_train), len(y_valid), len(y_test)],
    "share": [len(y_train) / len(y), len(y_valid) / len(y), len(y_test) / len(y)],
    "positives": [y_train.sum(), y_valid.sum(), y_test.sum()],
    "positive_rate": [y_train.mean(), y_valid.mean(), y_test.mean()],
}, index=["train", "validation", "test"]).round(4)
split_summary

In [ ]:
train_idx, valid_idx, test_idx = set(X_train.index), set(X_valid.index), set(X_test.index)

assert not train_idx & valid_idx and not train_idx & test_idx and not valid_idx & test_idx
assert len(train_idx | valid_idx | test_idx) == len(X)
assert X_train.index.equals(y_train.index) and X_valid.index.equals(y_valid.index) and X_test.index.equals(y_test.index)
print("Splits are disjoint, cover every row, and X/y indices are aligned.")

**Interpretation.** The positive-class rate is essentially identical across train, validation, and test, so differences between them later reflect the models rather than an uneven split. A random split measures performance on clients drawn from the same historical campaigns; it does not test generalization to future campaigns. The dataset is also a historical benchmark (2008–2010 Portuguese bank campaigns), so scores here are not evidence of current banking performance.

### 3.1 Is `"unknown"` informative? (training rows only)

This check uses the target, so it is computed on the training split only; validation and test labels do not inform the cleaning discussion.

In [ ]:
# Subscription rate when a field is "unknown" vs. known, using training rows only.
rows = []
for col in unknown_summary.index[unknown_summary["unknown_count"] > 0]:
    is_unknown = X_train[col] == "unknown"
    rows.append({
        "column": col,
        "train_unknown_rows": int(is_unknown.sum()),
        "positive_rate_unknown": y_train[is_unknown].mean(),
        "positive_rate_known": y_train[~is_unknown].mean(),
    })
pd.DataFrame(rows).set_index("column").round(4)

**Interpretation.** On training rows, clients with `default = "unknown"` subscribe at about 5.3%, versus 12.9% when `default` is known. Unknown `education` and `marital` values have higher rates than known values (about 14.6% and 19.0% versus 11%), although `marital` has only 58 unknown training rows. `"unknown"` therefore carries signal, which supports keeping it as its own category rather than imputing a mode.

## 4. Leakage-safe preprocessing

Learned preprocessing is fitted on `X_train` only and then applied unchanged to validation and test:

- **One-hot encoding** for all 10 nominal categorical columns, with `handle_unknown="ignore"` so a category that never appears in training becomes an all-zero row instead of an error.
- **Standard scaling** of numeric columns for the MLP only. Gradient-based networks train poorly when inputs have very different scales (for example `nr.employed` around 5,000 versus `pdays_recorded` in {0, 1}). Tree splits depend only on the ordering of values, so the GBDT input keeps the raw numeric values.

Both inputs contain exactly the same columns and rows; the only model-specific difference is numeric scaling. Each `preprocessing_sec` value times the full trio for that preprocessor — `fit_transform(X_train)` plus `transform(X_valid)` plus `transform(X_test)` — and is recorded separately from model fitting time.

In [ ]:
def make_preprocessor(scale_numeric: bool) -> ColumnTransformer:
    numeric_step = StandardScaler() if scale_numeric else "passthrough"
    return ColumnTransformer(
        transformers=[
            ("num", numeric_step, NUMERIC_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
        ],
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")


def fit_transform_splits(preprocessor: ColumnTransformer):
    """Fit on training rows only; reuse the fitted transformer for validation and test."""
    start = time.perf_counter()
    train_t = preprocessor.fit_transform(X_train)
    valid_t = preprocessor.transform(X_valid)
    test_t = preprocessor.transform(X_test)
    return train_t, valid_t, test_t, time.perf_counter() - start


tree_preprocessor = make_preprocessor(scale_numeric=False)
mlp_preprocessor = make_preprocessor(scale_numeric=True)

X_train_tree, X_valid_tree, X_test_tree, tree_preprocess_sec = fit_transform_splits(tree_preprocessor)
X_train_mlp, X_valid_mlp, X_test_mlp, mlp_preprocess_sec = fit_transform_splits(mlp_preprocessor)

PREPROCESSING_TIME_SEC = {"GBDT (one-hot only)": tree_preprocess_sec, "MLP (one-hot + scaling)": mlp_preprocess_sec}

pd.DataFrame({
    "train": [X_train_tree.shape, X_train_mlp.shape],
    "validation": [X_valid_tree.shape, X_valid_mlp.shape],
    "test": [X_test_tree.shape, X_test_mlp.shape],
    "preprocessing_sec": [round(tree_preprocess_sec, 4), round(mlp_preprocess_sec, 4)],
}, index=["GBDT input", "MLP input"])

In [ ]:
# 1. Both models see the same encoded columns (same underlying information).
assert list(X_train_tree.columns) == list(X_train_mlp.columns)

# 2. No missing values after preprocessing.
for frame in [X_train_tree, X_valid_tree, X_test_tree, X_train_mlp, X_valid_mlp, X_test_mlp]:
    assert not frame.isna().any().any()

# 3. Tree numeric columns are unchanged; MLP numeric columns are standardized with training statistics.
assert np.allclose(X_train_tree[NUMERIC_FEATURES].to_numpy(), X_train[NUMERIC_FEATURES].to_numpy())

scaling_check = pd.DataFrame({
    "train_mean": X_train_mlp[NUMERIC_FEATURES].mean(),
    "train_std": X_train_mlp[NUMERIC_FEATURES].std(ddof=0),
    "valid_mean": X_valid_mlp[NUMERIC_FEATURES].mean(),
    "test_mean": X_test_mlp[NUMERIC_FEATURES].mean(),
}).round(4)

n_onehot = len(X_train_tree.columns) - len(NUMERIC_FEATURES)
print(f"Encoded features: {X_train_tree.shape[1]} ({len(NUMERIC_FEATURES)} numeric + {n_onehot} one-hot)")
print("Categories with no training rows (ignored if seen later):",
      [f"{col}={val}" for col in CATEGORICAL_FEATURES for val in X[col].unique()
       if val not in set(X_train[col])] or "none")
scaling_check

**Interpretation.** Both models receive the same encoded features, and nothing is missing. The scaled MLP features have mean 0 and standard deviation 1 on the training rows only; validation and test means are close to, but not exactly, 0. That small difference is expected: it shows the scaler reused training statistics instead of being refit on held-out data.

## 5. Gradient-boosted decision trees (XGBoost)

All runs use `xgboost.XGBClassifier(tree_method="hist")` on the GBDT input from Section 4 (one-hot, unscaled numerics), the same training rows, and the same validation rows.

**Fixed settings**

- `n_estimators = 5000` as an upper limit, with `early_stopping_rounds = 50` and `eval_metric = "logloss"` set in the constructor.
- `eval_set = [(train), (validation)]`: binary log loss is tracked on both, and validation, listed last, controls early stopping.
- `random_state = 42`; all other hyperparameters at XGBoost defaults.

**Two different metrics.** Validation *log loss* decides when to stop adding trees within a run. Validation *Average Precision* (from `predict_proba`, via `average_precision_score`) compares configurations and selects the final model. The test set is not used here.

**Controlled experiments (8 configurations)**

| Phase | Varied | Held fixed |
|---|---|---|
| A | `learning_rate` ∈ {0.01, 0.1, 0.3} | `max_depth = 4`, `subsample = 1.0` |
| B | `max_depth` ∈ {2, 4, 6, 8} | best learning rate from A, `subsample = 1.0` |
| C | `subsample` ∈ {1.0, 0.8, 0.6} (regularization) | best learning rate and depth |

Phases share their reference configuration (for example, depth 4 in phase B is the phase A run), so each comparison changes one setting at a time.

In [ ]:
GBDT_N_ESTIMATORS = 5000          # upper limit; early stopping decides the actual number of trees
GBDT_EARLY_STOPPING_ROUNDS = 50   # stop after 50 rounds without validation log-loss improvement
GBDT_BASE_PARAMS = {"learning_rate": 0.1, "max_depth": 4, "subsample": 1.0}

gbdt_models: dict[str, xgb.XGBClassifier] = {}
gbdt_records: list[dict] = []


def fit_gbdt(config_id: str, phase: str, **overrides) -> dict:
    """Train one XGBoost configuration with validation-based early stopping and record the result."""
    params = {**GBDT_BASE_PARAMS, **overrides}
    model = xgb.XGBClassifier(
        tree_method="hist",
        n_estimators=GBDT_N_ESTIMATORS,
        early_stopping_rounds=GBDT_EARLY_STOPPING_ROUNDS,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **params,
    )
    start = time.perf_counter()
    # Validation is listed last, so it controls early stopping.
    model.fit(X_train_tree, y_train, eval_set=[(X_train_tree, y_train), (X_valid_tree, y_valid)], verbose=False)
    fit_sec = time.perf_counter() - start

    history = model.evals_result()
    best = model.best_iteration
    trees_built = len(history["validation_1"]["logloss"])
    train_loss = history["validation_0"]["logloss"][best]
    valid_loss = history["validation_1"]["logloss"][best]
    # With early stopping, predict_proba uses the trees up to best_iteration.
    valid_ap = average_precision_score(y_valid, model.predict_proba(X_valid_tree)[:, 1])

    record = {
        "config": config_id,
        "phase": phase,
        **params,
        "best_iteration": best,
        "trees_built": trees_built,
        # Cap-limited only if the run ended at the cap before early stopping could trigger.
        "hit_cap": trees_built == GBDT_N_ESTIMATORS and trees_built - 1 - best < GBDT_EARLY_STOPPING_ROUNDS,
        "train_logloss": train_loss,
        "valid_logloss": valid_loss,
        "logloss_gap": valid_loss - train_loss,
        "valid_AP": valid_ap,
        "fit_sec": fit_sec,
    }
    # Re-running a phase replaces its earlier record and model under the same ID.
    gbdt_records[:] = [r for r in gbdt_records if r["config"] != config_id]
    gbdt_models[config_id] = model
    gbdt_records.append(record)
    return record


def gbdt_table(records: list[dict]) -> pd.DataFrame:
    return pd.DataFrame(records).set_index("config").round(4)

### 5.1 Phase A — learning rate

In [ ]:
for i, lr in enumerate([0.01, 0.1, 0.3], start=1):
    fit_gbdt(f"G{i}", "A: learning rate", learning_rate=lr)

phase_a = [r for r in gbdt_records if r["phase"] == "A: learning rate"]
best_lr = max(phase_a, key=lambda r: r["valid_AP"])["learning_rate"]
print(f"Learning rate carried forward (best validation AP in phase A): {best_lr}")
gbdt_table(phase_a)

### 5.2 Phase B — tree depth

In [ ]:
for i, depth in enumerate([2, 6, 8], start=4):
    fit_gbdt(f"G{i}", "B: max depth", learning_rate=best_lr, max_depth=depth)

# Depth 4 at the chosen learning rate was already trained in phase A and is reused as the reference.
depth_comparison = [r for r in gbdt_records if r["learning_rate"] == best_lr and r["subsample"] == 1.0]
best_depth = max(depth_comparison, key=lambda r: r["valid_AP"])["max_depth"]
print(f"Max depth carried forward (best validation AP among depths): {best_depth}")
gbdt_table(sorted(depth_comparison, key=lambda r: r["max_depth"]))

### 5.3 Phase C — row subsampling

`subsample` trains each tree on a random fraction of the training rows. It is the regularization control varied here: adding randomness makes individual trees less able to fit noise in the training set.

In [ ]:
for i, subsample in enumerate([0.8, 0.6], start=7):
    fit_gbdt(f"G{i}", "C: subsample", learning_rate=best_lr, max_depth=best_depth, subsample=subsample)

# subsample = 1.0 at the chosen learning rate and depth is the reference from earlier phases.
subsample_comparison = [r for r in gbdt_records if r["learning_rate"] == best_lr and r["max_depth"] == best_depth]
gbdt_table(sorted(subsample_comparison, key=lambda r: -r["subsample"]))

### 5.4 All configurations and selection

The final GBDT is the configuration with the highest validation AP.

In [ ]:
gbdt_results = gbdt_table(gbdt_records)
# Select on unrounded validation AP from gbdt_records (gbdt_table rounds to 4 decimals).
selected_record = max(gbdt_records, key=lambda r: r["valid_AP"])
GBDT_SELECTED_ID = selected_record["config"]
gbdt_selected = gbdt_models[GBDT_SELECTED_ID]
GBDT_FIT_TIME_SEC = selected_record["fit_sec"]

print(f"Configurations tried: {len(gbdt_results)}")
print(f"Runs that reached the n_estimators cap without early stopping: {int(gbdt_results['hit_cap'].sum())}")
print(f"Selected GBDT: {GBDT_SELECTED_ID} "
      f"(learning_rate={selected_record['learning_rate']}, "
      f"max_depth={selected_record['max_depth']}, "
      f"subsample={selected_record['subsample']}) "
      f"— validation AP {selected_record['valid_AP']:.4f}")
print(f"Validation AP range across configurations: "
      f"{min(r['valid_AP'] for r in gbdt_records):.4f} – "
      f"{max(r['valid_AP'] for r in gbdt_records):.4f}")
gbdt_results



### 5.5 Learning curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = {0.01: "tab:blue", 0.1: "tab:orange", 0.3: "tab:green"}
for record in phase_a:
    model = gbdt_models[record["config"]]
    history = model.evals_result()
    lr = record["learning_rate"]
    rounds = np.arange(1, len(history["validation_1"]["logloss"]) + 1)  # 1-based so round 1 shows on a log axis
    ax.plot(rounds, history["validation_0"]["logloss"], color=colors[lr], linestyle="--", alpha=0.7, label=f"train, lr={lr}")
    ax.plot(rounds, history["validation_1"]["logloss"], color=colors[lr], label=f"validation, lr={lr}")
    ax.scatter(record["best_iteration"] + 1, record["valid_logloss"], color=colors[lr], zorder=5)

ax.set_xscale("log")
ax.set_ylim(0.22, 0.40)
ax.set_xlabel("Number of trees (log scale)")
ax.set_ylabel("Binary log loss")
ax.set_title(f"XGBoost learning-rate comparison (max_depth={GBDT_BASE_PARAMS['max_depth']}); dots = best iteration")
ax.legend(ncol=2, fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("fig_gbdt_learning_rates.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
history = gbdt_selected.evals_result()
best_it = gbdt_selected.best_iteration

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(history["validation_0"]["logloss"], label="training")
ax.plot(history["validation_1"]["logloss"], label="validation")
ax.axvline(best_it, color="gray", linestyle=":", label=f"best iteration = {best_it}")
ax.set_ylim(0.22, 0.40)
ax.set_xlabel("Boosting round")
ax.set_ylabel("Binary log loss")
ax.set_title(f"Selected GBDT ({GBDT_SELECTED_ID}): training vs. validation log loss")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("fig_gbdt_selected_logloss.png", dpi=200, bbox_inches="tight")
plt.show()

### 5.6 Feature importance

Importance measure: **total gain**, the sum of improvements in the training objective from splits on a feature across the trees used for prediction (boosting rounds `0` through `best_iteration`, inclusive), shown as a share of that total. One-hot encoding splits each categorical variable into separate columns (for example `poutcome_success`).



In [ ]:
# Importance over trees used for prediction only: Tree index is 0-based and
# best_iteration is inclusive, matching predict_proba under early stopping.
# Use total_gain (sum of split gains), not average gain per split.
booster = gbdt_selected.get_booster()
best_it = gbdt_selected.best_iteration
trees = booster.trees_to_dataframe()
split_gains = trees.loc[(trees["Tree"] <= best_it) & (trees["Feature"] != "Leaf")]
gain = split_gains.groupby("Feature")["Gain"].sum()
gain_share = (gain / gain.sum()).sort_values(ascending=False)
TOP_N = 15

fig, ax = plt.subplots(figsize=(8, 6))
gain_share.head(TOP_N).iloc[::-1].plot.barh(ax=ax, color="tab:blue")
ax.set_xlabel("Share of total gain")
ax.set_title(f"Selected GBDT ({GBDT_SELECTED_ID}): top {TOP_N} features by total gain")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig("fig_gbdt_feature_importance.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"Trees included: 0..{best_it} (best_iteration inclusive); "
      f"booster also has {booster.num_boosted_rounds() - (best_it + 1)} early-stopping extra rounds not used here")
print(f"Features used in at least one split: {len(gain)} of {X_train_tree.shape[1]}")
print(f"Top feature share: {gain_share.index[0]} = {gain_share.iloc[0]:.4f}")
gain_share.head(TOP_N).round(4).to_frame("gain_share")



**Interpretation.**

- **Learning rate mainly changes how fast boosting converges, not where it ends up.** All three rates reach similar validation AP (about 0.488–0.491). The best validation loss is reached after about 1,300 trees at `0.01`, about 180 at `0.1`, and about 50 at `0.3`. Larger steps reach the minimum sooner and then overfit sooner: at `0.3`, training loss keeps falling well after validation loss has bottomed out, and early stopping ends the run. The AP edge of `0.01` over the other rates is only about 0.001–0.002 — within noise — but under the selection rule it is the rate carried into phases B and C.
- **Depth controls the bias–variance trade-off.** Depth 2 has the highest training and validation loss and needs more than 3,000 trees at this learning rate: it underfits (high bias) and converges slowly. As depth increases, training loss falls faster than validation loss, so the train–validation gap widens from about 0.007 at depth 4 to about 0.03 at depth 8 (higher variance). Validation AP peaks at depth 4–6 and drops at depth 8.
- **Row subsampling has a small regularizing effect.** With `subsample` 0.8 or 0.6, the train–validation gap shrinks slightly and validation AP rises slightly. In this Colab run, `subsample = 0.8` has the highest validation AP among the eight configurations under the notebook's unrounded-`valid_AP` selection rule; another environment may select a different configuration when gaps are this small.
- **The differences are small.** Validation AP varies by only about 0.01 across all configurations. The validation set has about 700 positives, so gaps this small may partly reflect sampling noise; the selected configuration is a reasonable choice under a fixed rule rather than proof that it is uniquely best.
- **The selected model's curves** show both losses falling quickly over the first ~200 trees. After roughly 400 trees validation loss is almost flat while training loss keeps decreasing, opening a small gap; early stopping keeps the trees up to the best validation iteration.
- **Feature importance is dominated by macroeconomic indicators**, especially `nr.employed` (number employed, about half of total gain among trees through `best_iteration`), followed by other macros (`euribor3m`, `cons.conf.idx`, `emp.var.rate`), `pdays`, and campaign-history features such as `poutcome_success`. The macro indicators are strongly correlated with each other and change with the calendar period of the campaign, so gain can be split among them somewhat arbitrarily, and part of their importance reflects *when* a call happened. This is an association learned by the model, not a causal effect, and it is another reason a random split overstates performance on future campaigns.

## 6. Multi-layer perceptron (scikit-learn)

All runs use `sklearn.neural_network.MLPClassifier(solver="adam")` on the MLP input from Section 4 (one-hot categoricals plus numeric features standardized with training statistics), the same training rows, and the same validation rows as the GBDT experiments.

**Fixed settings**

- `max_iter = 200` epochs (scikit-learn's default) as the training budget, identical for every configuration.
- `early_stopping = False`, as recommended in the handout: all training rows are used for weight updates, and the external validation set is used only to compare configurations. Training can still end before `max_iter`: scikit-learn stops when the training loss improves by less than `tol = 1e-4` for 10 consecutive epochs, which counts as converged. A run that uses the full budget without meeting that criterion raises a `ConvergenceWarning`; these warnings are captured, printed, and recorded per run in the `converged` column.
- `activation = "relu"`, `batch_size = "auto"` (200 rows), `random_state = 42`; all other hyperparameters at scikit-learn defaults.

**What is measured.** `loss_curve_` is the training loss per epoch; it includes the L2 penalty, so its level depends on `alpha`. After training, `train_logloss` and `valid_logloss` are computed from `predict_proba` without the penalty, which makes them comparable across configurations and with the GBDT table. Validation Average Precision selects the final configuration. The test set is not used here.

**Controlled experiments (8 configurations, the same number as for GBDT)**

| Phase | Varied | Held fixed |
|---|---|---|
| A | `hidden_layer_sizes` ∈ {(64,), (128, 64), (128, 64, 32)} | `learning_rate_init = 0.001`, `alpha = 0.0001` (scikit-learn defaults) |
| B | `learning_rate_init` ∈ {0.0001, 0.001, 0.01} | best architecture from A, `alpha = 0.0001` |
| C | `alpha` ∈ {0.0001, 0.01, 0.1, 1.0} (L2 regularization) | best architecture and learning rate |

As for GBDT, phases share their reference configuration, so each comparison changes one setting at a time.

In [ ]:
MLP_MAX_ITER = 200   # epoch budget, identical for every configuration
MLP_BASE_PARAMS = {"hidden_layer_sizes": (64,), "learning_rate_init": 0.001, "alpha": 0.0001}

mlp_models: dict[str, MLPClassifier] = {}
mlp_records: list[dict] = []


def fit_mlp(config_id: str, phase: str, **overrides) -> dict:
    """Train one MLP configuration for the fixed epoch budget and record the result."""
    params = {**MLP_BASE_PARAMS, **overrides}
    model = MLPClassifier(
        solver="adam",
        activation="relu",
        max_iter=MLP_MAX_ITER,
        early_stopping=False,
        random_state=RANDOM_STATE,
        **params,
    )
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        start = time.perf_counter()
        model.fit(X_train_mlp, y_train)
        fit_sec = time.perf_counter() - start

    # Report convergence warnings explicitly and pass any other warning through unchanged.
    convergence_warnings = [w for w in caught if issubclass(w.category, ConvergenceWarning)]
    for w in caught:
        if issubclass(w.category, ConvergenceWarning):
            print(f"{config_id}: ConvergenceWarning — {w.message}")
        else:
            warnings.warn_explicit(w.message, w.category, w.filename, w.lineno)

    train_proba = model.predict_proba(X_train_mlp)[:, 1]
    valid_proba = model.predict_proba(X_valid_mlp)[:, 1]
    train_loss = log_loss(y_train, train_proba)
    valid_loss = log_loss(y_valid, valid_proba)

    record = {
        "config": config_id,
        "phase": phase,
        **params,
        "epochs": model.n_iter_,
        "converged": not convergence_warnings,
        "final_train_loss": model.loss_curve_[-1],
        "train_logloss": train_loss,
        "valid_logloss": valid_loss,
        "logloss_gap": valid_loss - train_loss,
        "valid_AP": average_precision_score(y_valid, valid_proba),
        "fit_sec": fit_sec,
    }
    # Re-running a phase replaces its earlier record and model under the same ID.
    mlp_records[:] = [r for r in mlp_records if r["config"] != config_id]
    mlp_models[config_id] = model
    mlp_records.append(record)
    return record


def mlp_table(records: list[dict]) -> pd.DataFrame:
    return pd.DataFrame(records).set_index("config").round(4)

### 6.1 Phase A — architecture

Three architectures of increasing width and depth: one hidden layer of 64 units, two layers (128, 64), and three layers (128, 64, 32).

In [ ]:
for i, layers in enumerate([(64,), (128, 64), (128, 64, 32)], start=1):
    fit_mlp(f"M{i}", "A: architecture", hidden_layer_sizes=layers)

phase_a_mlp = [r for r in mlp_records if r["phase"] == "A: architecture"]
best_layers = max(phase_a_mlp, key=lambda r: r["valid_AP"])["hidden_layer_sizes"]
print(f"Architecture carried forward (best validation AP in phase A): {best_layers}")
mlp_table(phase_a_mlp)

### 6.2 Phase B — initial learning rate

`learning_rate_init` is Adam's base step size. The 0.001 run at the chosen architecture is the reference from phase A.

In [ ]:
for i, lr in enumerate([0.0001, 0.01], start=4):
    fit_mlp(f"M{i}", "B: learning rate", hidden_layer_sizes=best_layers, learning_rate_init=lr)

lr_comparison = [r for r in mlp_records
                 if r["hidden_layer_sizes"] == best_layers and r["alpha"] == MLP_BASE_PARAMS["alpha"]]
best_lr_init = max(lr_comparison, key=lambda r: r["valid_AP"])["learning_rate_init"]
print(f"Learning rate carried forward (best validation AP in phase B): {best_lr_init}")
mlp_table(sorted(lr_comparison, key=lambda r: r["learning_rate_init"]))

### 6.3 Phase C — L2 regularization (`alpha`)

`alpha` scales an L2 penalty on the network weights that is added to the training loss. Larger values push weights toward zero and limit how closely the network can fit the training rows.

In [ ]:
for i, alpha in enumerate([0.01, 0.1, 1.0], start=6):
    fit_mlp(f"M{i}", "C: L2 alpha", hidden_layer_sizes=best_layers, learning_rate_init=best_lr_init, alpha=alpha)

# alpha = 0.0001 at the chosen architecture and learning rate is the reference from earlier phases.
alpha_comparison = [r for r in mlp_records
                    if r["hidden_layer_sizes"] == best_layers and r["learning_rate_init"] == best_lr_init]
mlp_table(sorted(alpha_comparison, key=lambda r: r["alpha"]))

### 6.4 All configurations and selection

The final MLP is the configuration with the highest validation AP.

In [ ]:
mlp_results = mlp_table(mlp_records)
selected_mlp_record = max(mlp_records, key=lambda r: r["valid_AP"])
MLP_SELECTED_ID = selected_mlp_record["config"]
mlp_selected = mlp_models[MLP_SELECTED_ID]
MLP_FIT_TIME_SEC = selected_mlp_record["fit_sec"]

print(f"Configurations tried: {len(mlp_results)}")
print(f"Runs that used the full {MLP_MAX_ITER}-epoch budget without converging: "
      f"{int((~mlp_results['converged']).sum())}")
print(f"Selected MLP: {MLP_SELECTED_ID} "
      f"(hidden_layer_sizes={selected_mlp_record['hidden_layer_sizes']}, "
      f"learning_rate_init={selected_mlp_record['learning_rate_init']}, "
      f"alpha={selected_mlp_record['alpha']}) "
      f"— validation AP {selected_mlp_record['valid_AP']:.4f}")
print(f"Validation AP range across configurations: "
      f"{min(r['valid_AP'] for r in mlp_records):.4f} – "
      f"{max(r['valid_AP'] for r in mlp_records):.4f}")
mlp_results

### 6.5 Validation performance by architecture and learning curves

The first figure compares the phase A architectures (validation AP, and training versus validation log loss). The second shows the training-loss curves (`loss_curve_`) for the three learning rates at the chosen architecture. The third shows the selected model's training-loss curve, with its final training and validation log loss for reference.

In [ ]:
arch_df = mlp_table(phase_a_mlp)
arch_labels = [str(layers) for layers in arch_df["hidden_layer_sizes"]]
positions = np.arange(len(arch_labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(positions, arch_df["valid_AP"], color="tab:blue")
for x, ap in zip(positions, arch_df["valid_AP"]):
    axes[0].annotate(f"{ap:.3f}", (x, ap), ha="center", va="bottom")
axes[0].set_ylabel("Validation AP")
axes[0].set_title("Validation AP by architecture")

width = 0.38
axes[1].bar(positions - width / 2, arch_df["train_logloss"], width, label="training")
axes[1].bar(positions + width / 2, arch_df["valid_logloss"], width, label="validation")
axes[1].set_ylabel("Log loss (no L2 penalty)")
axes[1].set_title("Training vs. validation log loss by architecture")
axes[1].legend()

for ax in axes:
    ax.set_xticks(positions, arch_labels)
    ax.set_xlabel("hidden_layer_sizes")
    ax.grid(axis="y", alpha=0.3)
fig.suptitle(f"MLP phase A (learning_rate_init = {MLP_BASE_PARAMS['learning_rate_init']}, "
             f"alpha = {MLP_BASE_PARAMS['alpha']}, max_iter = {MLP_MAX_ITER})")
fig.tight_layout()
fig.savefig("fig_mlp_architectures.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for record in sorted(lr_comparison, key=lambda r: r["learning_rate_init"]):
    curve = mlp_models[record["config"]].loss_curve_
    status = "converged" if record["converged"] else "budget reached"
    ax.plot(range(1, len(curve) + 1), curve,
            label=f"learning_rate_init = {record['learning_rate_init']} ({record['config']}, {status})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Training loss (log loss + L2 penalty)")
ax.set_title(f"MLP training loss by learning rate (hidden_layer_sizes = {best_layers})")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("fig_mlp_learning_rates.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
curve = mlp_selected.loss_curve_

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, len(curve) + 1), curve, label="training loss per epoch (loss_curve_)")
ax.axhline(selected_mlp_record["train_logloss"], color="tab:blue", linestyle=":",
           label=f"final training log loss = {selected_mlp_record['train_logloss']:.4f}")
ax.axhline(selected_mlp_record["valid_logloss"], color="tab:orange", linestyle="--",
           label=f"final validation log loss = {selected_mlp_record['valid_logloss']:.4f}")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title(f"Selected MLP ({MLP_SELECTED_ID}): training loss curve")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("fig_mlp_selected_loss.png", dpi=200, bbox_inches="tight")
plt.show()

**Interpretation.**

- **More capacity overfits badly under this budget.** At the default learning rate and `alpha`, the two deeper networks drive training log loss to about 0.04–0.05 while validation log loss rises to 0.92–1.13 and validation AP falls to 0.32–0.35. The single 64-unit layer overfits less (gap 0.09, AP 0.41). With `early_stopping=False`, nothing stops training once a network starts memorizing the training rows, so extra width and depth add variance rather than reduce bias. The (128, 64, 32) network converged after 150 epochs and still has the worst validation log loss: in scikit-learn, "converged" only means the training loss flattened, not that the model generalizes.
- **The learning rate decides how far training moves toward overfitting within 200 epochs.** At `0.01` the network reaches the lowest training loss of phase B (0.18) but the highest validation loss (0.48) and AP 0.35. At `0.0001` training is slower and the loss is still decreasing at epoch 200, but the train–validation gap almost disappears (0.008) and AP rises to 0.48. Unlike GBDT, where early stopping made the learning rate mainly a speed setting, here it changes the final model substantially.
- **L2 regularization closes the remaining gap.** With `alpha` 0.01 or 0.1 the gap shrinks to 0.003 and 0.0004, both runs converge after 135 epochs, and validation AP (0.4825 and 0.4810) is essentially tied with the unregularized `0.0001` run (0.4805). `alpha = 1.0` underfits slightly: both losses rise and AP drops to 0.467.
- **Convergence warnings and the `max_iter` budget.** Four of eight runs (M1, M2, M4, M5) used the full 200 epochs and raised a `ConvergenceWarning`: their training loss was still improving by more than `tol`. For M1, M2, and M5 more epochs would most likely increase overfitting, not help. M4's curve is nearly flat by epoch 200 and its AP is within 0.002 of the selected model. The budget is kept fixed so that all configurations are compared under the same training effort.
- **The differences among the best runs are small.** M4, M6, and M7 lie within 0.0015 AP of each other, well within sampling noise for about 700 validation positives. M6 is selected by the fixed rule rather than shown to be uniquely best. Its validation AP (0.4825) is about 0.01 below the selected GBDT (0.4923); the comparison on held-out data is in Step 4.
- **The selected model's curve** drops steeply over the first 10 epochs, then decreases slowly until the training-loss improvement falls below `tol` and training stops at epoch 135. Final training and validation log loss (0.2656 and 0.2684) are close, so the selected MLP shows no sign of overfitting.
- **Limitation of the sequential design.** The architecture was chosen at the default learning rate and `alpha`, where overfitting dominated. A deeper network might be competitive once trained with a smaller learning rate and L2 penalty; this was not tested in order to keep the search effort comparable to the GBDT experiments.